# Data Extraction (DATA_ONLY Mode) — Speculative Decoding Training

This notebook demonstrates how to extract hidden states from a verifier model using the
`SpeculativeDecodingTrainer` in **DATA_ONLY** mode on Red Hat OpenShift AI.

## What Does DATA_ONLY Mode Do?

DATA_ONLY mode extracts hidden states from the verifier model without performing any training.
The extracted hidden state tensors (`.safetensors` files) are written to the output PVC for
use in subsequent [TRAIN_ONLY](../train-only/) training runs. This is the first step of a
two-step workflow — extract once, then experiment with training hyperparameters many times.

## Two Extraction Methods

DATA_ONLY supports two methods for serving the verifier model during extraction:

| Method | CTR | Description |
|--------|-----|-------------|
| **Managed vLLM Sidecar** | `vllm-extract-cuda` | The SDK deploys and manages a vLLM sidecar alongside the job pod. Simplest option. |
| **External vLLM Endpoint** | `speculator-model-opt-cuda` | You provide a running vLLM server. Useful when you already have a vLLM deployment. |

This notebook includes both methods — run whichever suits your setup.

## Speculative Decoding Overview

Large language models generate tokens one at a time, and each token requires reading the entire
model from GPU memory — making inference **memory-bound**. Speculative decoding exploits this:
a small, fast **draft model** (~1.2 GB with Qwen3-0.6B) guesses the next several tokens, then the large **verifier
model** checks all guesses in a single forward pass. The output is mathematically identical to
normal decoding — no quality loss.

**Eagle3** is a draft model architecture that reads hidden states from four intermediate layers
of the verifier (not just the final logits), giving it richer context for more accurate
predictions.

## Dataset

This example uses the `ultrachat` built-in dataset (multi-turn conversational data).

## Hardware Requirements

The table below shows the **minimum** resources needed to run each method. See the
Configuration cell for recommended values that improve performance.

| Method | Component | GPU | CPU | Memory | Notes |
|--------|-----------|-----|-----|--------|-------|
| Managed sidecar (min) | vLLM sidecar | 1× GPU | 1 core | 48Gi | Minimum to load Qwen3-0.6B |
| Managed sidecar (recommended) | vLLM sidecar | 1× GPU | 4 cores | 96Gi | Headroom for KV cache |
| External vLLM (min) | Extraction pod | 0 GPU | 1 core | 16Gi | vLLM is external; no GPU needed |
| External vLLM (recommended) | Extraction pod | 0 GPU | 2 cores | 32Gi | Faster tokenization |

> **Note:** No training container is deployed in DATA_ONLY mode — only the extraction job runs.

## Setup

Install the Kubeflow SDK and import required dependencies.

In [ ]:
# Install the Kubeflow SDK from the OpenDataHub fork (includes SpeculativeDecodingTrainer)
!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/opendatahub-io/kubeflow-sdk.git@main

# Structured logging library (dependency for SDK progress tracking)
!pip install structlog

# Kubeflow Trainer API — provides TrainerClient, KubernetesBackendConfig, and job management
!pip install --no-cache-dir --force-reinstall --index-url https://pypi.org/simple kubeflow-trainer-api==2.3.0

In [ ]:
import os

import kubeflow

# Backend config tells the TrainerClient how to connect to the cluster
from kubeflow.common.types import KubernetesBackendConfig

# TrainerClient is the main entry point for submitting, monitoring, and deleting TrainJobs
from kubeflow.trainer import TrainerClient

# Name option lets you assign an explicit name to a TrainJob (otherwise auto-generated)
from kubeflow.trainer.options.common import Name

# Red Hat OpenShift AI extensions for speculative decoding training
from kubeflow.trainer.rhai import (
    SpeculativeDecodingTrainer,  # High-level trainer that wraps all four modes
    SpeculatorConfig,  # Fine-grained config: layer IDs, architecture, scheduler, etc.
    SpeculatorMode,  # Enum: DATA_ONLY, TRAIN_ONLY, OFFLINE, ONLINE
    SpeculatorType,  # Enum: EAGLE3 (currently the only supported type)
)

# Kubernetes Python client — used to configure API server auth
from kubernetes import client as k8s

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print("All imports successful")

In [ ]:
# Verify the SDK loaded correctly by inspecting the available enum values and defaults.
# This confirms that SpeculatorMode, SpeculatorType, and SpeculatorConfig are importable
# and behave as expected before proceeding to cluster authentication.
print(f"Modes: {[m.value for m in SpeculatorMode]}")
print(f"Types: {[t.value for t in SpeculatorType]}")
print(f"Config defaults: {SpeculatorConfig()}")
print("SDK ready")

## Authenticate to your OpenShift Cluster

The following environment variables are required for API authentication:

- `OPENSHIFT_API_URL` — your cluster API URL (e.g., `https://api.cluster.example.com:6443`)
- `NOTEBOOK_USER_TOKEN` — an access token for API calls

In OpenShift AI workbenches, these are often auto-set.

If they are not set in your environment, uncomment and populate the values in the next cell.

In [ ]:
# ============================================================================
# AUTHENTICATION
# ============================================================================
# If your workbench does not auto-populate these env vars, uncomment and fill them in:
#
# api_server = "https://api.your-cluster.example.com:6443"
# token = "sha256~your-token-here"

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")

if not api_server or not token:
    raise RuntimeError(
        "OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set. "
        "Either set them in your environment or uncomment the values above."
    )

# HuggingFace token — required for gated models; recommended for all models to avoid rate limits
HF_TOKEN = "<REPLACE WITH HF TOKEN>"

# Configure Kubernetes client
configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False  # Set to True if using trusted certificates
configuration.api_key = {"authorization": f"Bearer {token}"}

# ============================================================================
# PVC MOUNT PATHS
# ============================================================================
PVC_NAME = "shared"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        "Expected workbench PVC mount not found at: "
        f"{NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name/mount, update PVC_NAME/NOTEBOOK_SHARED_PATH.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# ============================================================================
# TRAINER CLIENT
# ============================================================================
trainer_client = TrainerClient(
    backend_config=KubernetesBackendConfig(client_configuration=configuration)
)

# ============================================================================
# CLUSTER TRAINING RUNTIMES (CTRs)
# ============================================================================
# DATA_ONLY supports two extraction methods, each using a different CTR:
#   Method 1 — Managed vLLM sidecar:  SDK deploys a vLLM sidecar in the pod
#   Method 2 — External vLLM endpoint: user provides a running vLLM server
DATA_EXTRACT_CTR = "vllm-extract-cuda"  # Method 1: managed vLLM sidecar
MODEL_OPT_CTR = "speculator-model-opt-cuda"  # Method 2: external vLLM endpoint

# Verify both CTRs exist on the cluster
available_runtimes = {r.name for r in trainer_client.list_runtimes()}
for ctr_name in [DATA_EXTRACT_CTR, MODEL_OPT_CTR]:
    status = (
        "Found" if ctr_name in available_runtimes else "WARNING: not found on cluster"
    )
    print(f"CTR '{ctr_name}': {status}")

print(f"\nAPI Server: {api_server}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount (SDK): {SDK_MOUNT_PATH}")

## (Optional) Download the Verifier Model

Pre-downloading the verifier model to the shared PVC speeds up pod startup — the
training pods will find the model on the PVC instead of downloading it from
HuggingFace during the job.

Skip this cell if the model is already on your PVC or if you prefer to let the
training pods download it automatically (requires `HF_TOKEN` in the `env` parameter).

In [ ]:
from huggingface_hub import snapshot_download

os.environ["HF_TOKEN"] = HF_TOKEN

model_id = "Qwen/Qwen3-0.6B"
local_dir = f"{NOTEBOOK_SHARED_PATH}/models/Qwen3-0.6B"

snapshot_download(model_id, local_dir=local_dir)
print(f"Model downloaded to {local_dir}")

## Configuration

The following constants are shared by both extraction methods. The verifier model is
[Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), a 28-layer transformer,
specified by its HuggingFace model ID. The training pods download the model
automatically — no manual pre-download is required (`HF_TOKEN` provides
authentication).

All output paths use **PVC URIs** (`pvc://<pvc-name>/<path>`), which the SDK
resolves to container mount paths internally.

`RUN_NAME` creates a namespace on the PVC for each experiment — change it to start
a fresh run without overwriting previous results.

In [ ]:
# Unique run identifier — namespaces all output paths on the PVC.
# Change this to start a fresh experiment without overwriting previous results.
RUN_NAME = "run-01"

# Set the Verifier Model for the training job.
VERIFIER_MODEL = "Qwen/Qwen3-0.6B"
# If you pre-downloaded the model to the PVC, use the PVC URI instead:
# VERIFIER_MODEL = f"pvc://{PVC_NAME}/models/Qwen3-0.6B"

# Eagle3 reads hidden states from 4 intermediate layers of the verifier.
# Qwen3-0.6B has 28 transformer layers (indexed 1-28).
# Layers chosen: early (2), mid (14), late (25), and final (28) — giving the
# draft model a spread of low-level, mid-level, and high-level representations.
TARGET_LAYER_IDS = [2, 14, 25, 28]

# Output directory on PVC — hidden states will be written to <output_dir>/hidden_states/
# This path is the same regardless of which extraction method you use.
DATA_ONLY_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}"

# ============================================================================
# METHOD 1 RESOURCES — Managed vLLM Sidecar
# ============================================================================
# Minimum resources to run the vLLM sidecar with Qwen3-0.6B.
# The sidecar is hard-limited to exactly 1 GPU — more raises a ValueError.
VLLM_RESOURCES = {
    "nvidia.com/gpu": 1,
    "cpu": "1",  # Recommended: "4" — faster tokenization and model loading
    "memory": "48Gi",  # Recommended: "96Gi" — more headroom for KV cache
}

# ============================================================================
# METHOD 2 CONFIGURATION — External vLLM Endpoint
# ============================================================================
# URL of your externally managed vLLM server.
# This must be a running vLLM instance serving the same verifier model (Qwen3-0.6B).
# The /v1 path exposes the OpenAI-compatible API that the SDK calls for extraction.
# Replace <namespace> with the namespace where your vLLM server is deployed.
VLLM_ENDPOINT = "http://vllm-svc.<namespace>.svc.cluster.local:8000/v1"

# Data extraction parameters (shared by both methods)
TOTAL_SEQ_LEN = 2048  # Maximum sequence length for extraction
MAX_SAMPLES = 500  # Cap on the number of dataset samples to process

print("Shared Configuration:")
print(f"  Run name:          {RUN_NAME}")
print(f"  Verifier model:    {VERIFIER_MODEL}")
print(f"  Target layers:     {TARGET_LAYER_IDS}")
print(f"  Output dir:        {DATA_ONLY_OUTPUT}")
print(f"  Sequence length:   {TOTAL_SEQ_LEN}")
print(f"  Max samples:       {MAX_SAMPLES}")
print(
    f"\nMethod 1 (sidecar):  {VLLM_RESOURCES['nvidia.com/gpu']} GPU,"
    f" {VLLM_RESOURCES['memory']} memory"
)
print(f"Method 2 (external): {VLLM_ENDPOINT}")

## Method 1: Managed vLLM Sidecar

The SDK deploys a managed vLLM sidecar alongside the extraction job pod. The sidecar
serves the verifier model, processes the dataset, and writes hidden state tensors to
the output PVC. The sidecar is automatically cleaned up when the job completes.

**Use this method when** you want the simplest setup — no external vLLM server needed.

**CTR:** `vllm-extract-cuda` — includes a vLLM sidecar container in the pod spec.

**Key parameters:**
- `vllm_resources` — GPU/CPU/memory for the managed vLLM sidecar
- `vllm_gpu_memory_utilization` — Fraction of GPU memory the sidecar can use (0.9 = 90%)
- `regenerate_responses` — When `True`, generates new responses from prompts before extraction
- `datagen_concurrency` — Number of concurrent data generation workers
- `hidden_states_dtype` — Data type for saved tensors (`bfloat16` saves disk space)

**Not needed**: `training_resources` or `vllm_endpoint`

In [ ]:
# Job name for Method 1 (managed sidecar)
DATA_JOB = f"eagle3-data-{RUN_NAME}"

# Configure the DATA_ONLY trainer with a managed vLLM sidecar.
# The SDK deploys a vLLM sidecar alongside the job pod for hidden state extraction.
data_only_sidecar = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.DATA_ONLY,
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL,  # HuggingFace model ID — downloaded automatically by the job
    dataset_name="ultrachat",  # Built-in multi-turn conversational dataset
    max_samples=MAX_SAMPLES,
    total_seq_len=TOTAL_SEQ_LEN,
    vllm_resources=VLLM_RESOURCES,  # GPU/CPU/memory for the managed vLLM sidecar
    vllm_gpu_memory_utilization=0.9,  # Let vLLM use 90% of GPU memory for KV cache
    regenerate_responses=True,  # Generate fresh responses from prompts
    enable_progression_tracking=True,
    packages_to_install=["speculators==0.6.0", "torchvision==0.24.0"],
    output_dir=DATA_ONLY_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
        datagen_concurrency=4,
        hidden_states_dtype="bfloat16",
    ),
    env={"HF_TOKEN": HF_TOKEN},
)

print("Method 1 — Managed vLLM Sidecar:")
print(f"  Job name:      {DATA_JOB}")
print(f"  Mode:          {data_only_sidecar.mode.value}")
print(f"  Verifier:      {data_only_sidecar.verifier_model}")
print(f"  Dataset:       {data_only_sidecar.dataset_name}")
print(f"  Max samples:   {data_only_sidecar.max_samples}")
print(f"  Target layers: {data_only_sidecar.config.target_layer_ids}")
print(f"  Output dir:    {data_only_sidecar.output_dir}")
print(f"  CTR:           {DATA_EXTRACT_CTR}")

In [ ]:
# Submit the DATA_ONLY TrainJob using the managed vLLM sidecar.
# Uses DATA_EXTRACT_CTR which includes a vLLM sidecar in the pod spec.
trainer_client.train(
    options=[Name(name=DATA_JOB)],
    trainer=data_only_sidecar,
    runtime=DATA_EXTRACT_CTR,
)

print(f"Method 1 job submitted: {DATA_JOB}")
print("\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={DATA_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the Method 1 job (Pending, Running, Succeeded, Failed).
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(DATA_JOB)

### Clean up before running Method 2

> **Important:** If you ran Method 1 above, you must delete the TrainJob and remove
> its output directory before running Method 2. Both methods write to the same
> `DATA_ONLY_OUTPUT` path. If hidden states from Method 1 already exist at that path,
> Method 2 will detect them and skip extraction — producing no new output.

Run the cell below to delete the Method 1 job and clean up its output. **Skip this
cell** if you did not run Method 1 or if you want to keep its output and only run
Method 2 with a different `RUN_NAME`.

In [ ]:
# Uncomment the lines below to delete Method 1's TrainJob and clean up its output.

# Delete the Method 1 TrainJob
# trainer_client.delete_job(DATA_JOB)
# print(f"Deleted TrainJob: {DATA_JOB}")

# Remove Method 1's output directory from the PVC so Method 2 starts fresh.
# WARNING: This permanently deletes the extracted hidden states from Method 1.
# import shutil
# output_path = f"{NOTEBOOK_SHARED_PATH}/speculator/{RUN_NAME}"
# shutil.rmtree(output_path, ignore_errors=True)
# print(f"Removed output directory: {output_path}")

## Method 2: External vLLM Endpoint

This method connects to a **self-managed external vLLM server** for hidden state
extraction. The SDK does not deploy a vLLM sidecar — you provide a `vllm_endpoint`
pointing to your own running vLLM instance.

**Use this method when** you already have a vLLM deployment (e.g., an OpenShift AI
model serving instance) and want to reuse it for extraction.

**CTR:** `speculator-model-opt-cuda` — no vLLM sidecar included.

**Prerequisites:**
- A running vLLM server serving the same verifier model (Qwen3-0.6B)
- The server must expose the OpenAI-compatible API (typically port 8000, path `/v1`)
- The server must be accessible from the training pods (e.g., via a Kubernetes service URL)
- The server must be in the **same namespace** and have access to the **same shared PVC**
  as the TrainJob — hidden states are written to the PVC and both the extraction job and
  vLLM server must see the same filesystem

**Key parameters:**
- `vllm_endpoint` — URL of your external vLLM server
- No `vllm_resources` needed (the external server handles its own resources)

In [ ]:
# Job name for Method 2 (external vLLM)
DATA_EXT_JOB = f"eagle3-data-ext-{RUN_NAME}"

# Configure the DATA_ONLY trainer with an external vLLM endpoint.
# The SDK connects to your running vLLM server for extraction — no sidecar is deployed.
data_only_external = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.DATA_ONLY,
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL,  # HuggingFace model ID — downloaded automatically by the job
    dataset_name="ultrachat",  # Built-in multi-turn conversational dataset
    max_samples=MAX_SAMPLES,
    total_seq_len=TOTAL_SEQ_LEN,
    hidden_states_path=f"{DATA_ONLY_OUTPUT}/hidden_states",
    vllm_endpoint=VLLM_ENDPOINT,  # External vLLM server URL (no sidecar deployed)
    regenerate_responses=True,  # Generate fresh responses from prompts
    enable_progression_tracking=True,
    packages_to_install=["speculators==0.6.0", "torchvision==0.24.0"],
    output_dir=DATA_ONLY_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
        datagen_concurrency=4,
        hidden_states_dtype="bfloat16",
    ),
    env={"HF_TOKEN": HF_TOKEN},
)

print("Method 2 — External vLLM Endpoint:")
print(f"  Job name:      {DATA_EXT_JOB}")
print(f"  Mode:          {data_only_external.mode.value}")
print(f"  Verifier:      {data_only_external.verifier_model}")
print(f"  vLLM endpoint: {data_only_external.vllm_endpoint}")
print(f"  Dataset:       {data_only_external.dataset_name}")
print(f"  Max samples:   {data_only_external.max_samples}")
print(f"  Target layers: {data_only_external.config.target_layer_ids}")
print(f"  Output dir:    {data_only_external.output_dir}")
print(f"  CTR:           {MODEL_OPT_CTR}")

In [ ]:
# Submit the DATA_ONLY TrainJob using the external vLLM endpoint.
# Uses MODEL_OPT_CTR — no SDK-managed sidecar needed.
trainer_client.train(
    options=[Name(name=DATA_EXT_JOB)],
    trainer=data_only_external,
    runtime=MODEL_OPT_CTR,
)

print(f"Method 2 job submitted: {DATA_EXT_JOB}")
print("\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={DATA_EXT_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the Method 2 job.
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(DATA_EXT_JOB)

## Cleanup

Delete the TrainJob when you are done. Uncomment the line below to delete.

In [ ]:
# Delete completed TrainJob(s) to free cluster resources (pods, volumes, etc.).
# Note: Deleting a job does NOT delete the output data on the PVC —
# hidden states remain available for the TRAIN_ONLY step.

# Uncomment the method you used:

# Method 1 (managed sidecar):
# trainer_client.delete_job(DATA_JOB)

# Method 2 (external vLLM):
# trainer_client.delete_job(DATA_EXT_JOB)

# print("TrainJob deleted.")

## Summary

This notebook extracted hidden states from Qwen3-0.6B using the `ultrachat` dataset.
The extracted data is stored on the PVC at
`pvc://<pvc-name>/speculator/<run-name>/hidden_states/` and can be reused across
multiple training runs.

Two extraction methods were demonstrated:
- **Method 1 (Managed vLLM Sidecar)** — simplest setup, SDK manages the vLLM sidecar
- **Method 2 (External vLLM Endpoint)** — reuses an existing vLLM deployment

Both methods produce identical output — the TRAIN_ONLY step works the same regardless
of which method was used.

### Next Steps

- **Train the draft model**: Run the [TRAIN_ONLY](../train-only/) notebook to train
  an Eagle3 draft model from the hidden states extracted in this step. You can iterate
  on training hyperparameters (`epochs`, `lr`, `num_layers`, etc.) without re-running
  the expensive extraction step.

### Alternative Modes

- [OFFLINE](../offline/) — Extract + train in a single job using an external vLLM endpoint
- [ONLINE](../online/) — Fully managed end-to-end extraction and training in one step